In [ ]:
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma



loader = PyPDFLoader('python-ai.pdf')
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
texts = splitter.split_documents(docs)


from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

collection_name = "my_rag_collection"


client = chromadb.Client()
print("Initialized in-memory ChromaDB client.")

try:


    _ = client.get_or_create_collection(name=collection_name)
    print(f"ChromaDB native client successfully got or created collection '{collection_name}' (in-memory).")
except Exception as e:
    raise RuntimeError(f"Failed to create/get ChromaDB collection with native client: {e}. This is unexpected for an in-memory client.") from e


vectorstore = Chroma(
    client=client,
    collection_name=collection_name,
    embedding_function=embeddings,
    create_collection_if_not_exists=False
)



vectorstore.add_documents(documents=texts)

results = vectorstore.similarity_search("What is ai?", k=3)

for r in results:
    print(r.page_content)
    print("---")

